In [ ]:
import torch
import matplotlib.pyplot as plt

In [ ]:
# get the data set of 32x32 RGB images from the CIFAR-10 dataset
from torchvision import datasets
from torchvision import transforms

# define where the data will be stored
data_path = '../../data/data-unversioned/p1ch7/'

# download the training data
cifar10 = datasets.CIFAR10(data_path, train = True, download = True)

# download the validation data
cifar10_val = datasets.CIFAR10(data_path, train = False, download = True)

# define the class names as a list (corresponds to class)
class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']


In [ ]:
# we have taken the means and variances from the previous notebook.
# however, the reality would be that we would need to transform to a 
# tensor first, perform the stack to collect all images, and then
# perform the view method followed by mean and std.
# Also, we would probably do this on the subset we are about to consider...

cifar10 = datasets.CIFAR10(
    data_path, train=True, download=False,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))
cifar10_val = datasets.CIFAR10(
    data_path, train=False, download=False,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))

In [ ]:
# now we are going to subset the data
# to just consider birds and planes
label_map = {0:0, 2:1}
class_names = ['airplane','bird']
cifar2 = [(img, label_map[label]) for img, label in cifar10 if label in [0,2]]
cifar2_val = [(img, label_map[label]) for img, label in cifar10_val if label in [0,2]]

In [ ]:
import torch.nn as nn
model = nn.Sequential(
    nn.Linear(3072,512), # 3072 = 3 x 32 x 32 for squeezed data, 512 is arbitrary
    nn.Tanh(),
    nn.Linear(512,2), # 2 outputs for the two possible states (i.e. using one-hot-encoding as a hidden target)
    nn.Softmax(dim=1) # convert to probabilities; apply across dim=1 b/c batches are dim 0
)

In [ ]:
# running the model based on random initialization
img, _ = cifar2[0]

plt.imshow(img.permute(1,2,0)) # place channels at end, as expected by imshow
plt.show()

My guess is that it's a bird. Note that this is post normalization, so imshow would ignore a lot of data.

In [ ]:
# use view method to push all data into a single dimension
# use unsqueeze 0 to add a singleton dimension in the first 
# dimension index, representing a batch.
# the model expects dimension 0 to be the index of data points
# and the next dimensions to represent the data points themselves
img_batch = img.view(-1).unsqueeze(0)

In [ ]:
out = model(img_batch)
out

Note that, although these outputs are technically probabilities, after training the model will be over-confident. There is a thing called Bayesian Neural Networks, which is out of scope of this book but worth looking into.

In [ ]:
# get the index (don't bother with element) that is associated with the maximum probability
_, index = torch.max(out, dim=1) # again batches are in dim=0, so we want max across categories in dim=1
class_names[index]

In [ ]:
# to incorporate the negative log likelihood cost function
# we are going to rewrite our model class structure
model = nn.Sequential(
    nn.Linear(3072, 512),
    nn.Tanh(),
    nn.Linear(512,2),
    nn.LogSoftmax(dim=1) # LogSoftmax is more numerically stable than Softmax
)

In [ ]:
# define the loss
loss = nn.NLLLoss()

In [ ]:
# test this
img, label = cifar2[0]
out = model(img.view(-1).unsqueeze(0))

loss(out, torch.tensor([label]))

In [ ]:
# now let us optimize
import torch.optim as optim

learning_rate = 1e-2

optimizer = optim.SGD(
    model.parameters(),
    lr = learning_rate
)

loss_fn = nn.NLLLoss()

n_epochs = 100

# DataLoader class helps with constructing batches from the full Dataset class
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle = True)

for epoch in range(n_epochs):
    for imgs, labels in train_loader:
        batch_size = imgs.shape[0]
        outputs = model(imgs.view(batch_size,-1))
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch: %d, Loss: %f" % (epoch, float(loss)))

In [ ]:
# generate the validation dataloader class
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64, shuffle=False)

correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in val_loader:
        batch_size = imgs.shape[0]
        outputs = model(imgs.view(batch_size,-1))
        _, predicted = torch.max(outputs, dim=1)
        total += labels.shape[0]
        correct += int((predicted == labels).sum())

print("Accuracy: %f", correct/total)

In [ ]:
# Let us try a deeper model!

model = nn.Sequential(
    nn.Linear(3072, 1024),
    nn.Tanh(),
    nn.Linear(1024,512),
    nn.Tanh(),
    nn.Linear(512,128),
    nn.Tanh(),
    nn.Linear(128,2),
    nn.LogSoftmax(dim=1) # LogSoftmax is more numerically stable than Softmax
)

# define the loss
loss = nn.NLLLoss()

### Optimization ###
learning_rate = 1e-2

optimizer = optim.SGD(
    model.parameters(),
    lr = learning_rate
)

loss_fn = nn.NLLLoss()

n_epochs = 100

# DataLoader class helps with constructing batches from the full Dataset class
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle = True)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64, shuffle = False)

train_loss_data = []
val_loss_data = []

for epoch in range(n_epochs):
    accum_train_loss = 0
    for imgs, labels in train_loader:
        # calculate train loss
        batch_size = imgs.shape[0]
        outputs = model(imgs.view(batch_size,-1))
        loss = loss_fn(outputs, labels)
        accum_train_loss += loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss_data.append(accum_train_loss)
    with torch.no_grad():
        accum_val_loss = 0
        for imgs, labels in val_loader:
            batch_size = imgs.shape[0]
            outputs = model(imgs.view(batch_size,-1))
            loss = loss_fn(outputs, labels)
            accum_val_loss += loss
        val_loss_data.append(accum_val_loss)
    print("Epoch: %d, Loss: %f" % (epoch, float(loss)))

In [ ]:
train_loss_data = [data.item() for data in train_loss_data]
train_loss_data

In [ ]:
val_loss_data = [data.item() for data in val_loss_data]
val_loss_data

In [ ]:
plt.figure()
plt.plot(train_loss_data,label = "Train")
plt.plot(val_loss_data, label = "Validation")
plt.xlabel("Epochs")
plt.ylabel('Loss')
plt.legend()
plt.show()

Let's try a different optimizer :) 

In [ ]:
# Let us try a deeper model!

model = nn.Sequential(
    nn.Linear(3072, 1024),
    nn.Tanh(),
    nn.Linear(1024,512),
    nn.Tanh(),
    nn.Linear(512,128),
    nn.Tanh(),
    nn.Linear(128,2),
    nn.LogSoftmax(dim=1) # LogSoftmax is more numerically stable than Softmax
)

# define the loss
loss = nn.NLLLoss()

### Optimization ###
learning_rate = 1e-4

optimizer = optim.Adam(
    model.parameters(),
    lr = learning_rate
)

loss_fn = nn.NLLLoss()

n_epochs = 100

# DataLoader class helps with constructing batches from the full Dataset class
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=100, shuffle = True)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=100, shuffle = False)

train_loss_data = []
val_loss_data = []

for epoch in range(n_epochs):
    accum_train_loss = 0
    for imgs, labels in train_loader:
        # calculate train loss
        batch_size = imgs.shape[0]
        outputs = model(imgs.view(batch_size,-1))
        loss = loss_fn(outputs, labels)
        accum_train_loss += loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss_data.append(float(accum_train_loss/len(train_loader.dataset)))

    with torch.no_grad():
        accum_val_loss = 0
        for imgs, labels in val_loader:
            batch_size = imgs.shape[0]
            outputs = model(imgs.view(batch_size,-1))
            loss = loss_fn(outputs, labels)
            accum_val_loss += loss
        val_loss_data.append(float(accum_val_loss/len(val_loader.dataset)))
        
    print("Epoch: %d, Train Loss: %f, Validation Loss: %f" % (epoch, accum_train_loss, accum_val_loss))

In [ ]:
plt.figure()
plt.plot(train_loss_data,label = "Train")
plt.plot(val_loss_data, label = "Validation")
plt.xlabel("Epochs")
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# check the number of parameters
numel_list = [p.numel() for p in model.parameters() if p.requires_grad == True]

sum(numel_list), numel_list

In [ ]:
len(cifar2)

Let's try a much smaller network to see if we can stop overfitting!

In [ ]:
# Let us try a deeper model!

model = nn.Sequential(
    nn.Linear(3072, 512),
    nn.Tanh(),
    nn.Linear(512,2),
    nn.LogSoftmax(dim=1) # LogSoftmax is more numerically stable than Softmax
)

# define the loss
loss = nn.NLLLoss()

### Optimization ###
learning_rate = 1e-4

optimizer = optim.Adam(
    model.parameters(),
    lr = learning_rate
)

loss_fn = nn.NLLLoss()

n_epochs = 100

# DataLoader class helps with constructing batches from the full Dataset class
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=100, shuffle = True)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=100, shuffle = False)

train_loss_data = []
val_loss_data = []

for epoch in range(n_epochs):
    accum_train_loss = 0
    for imgs, labels in train_loader:
        # calculate train loss
        batch_size = imgs.shape[0]
        outputs = model(imgs.view(batch_size,-1))
        loss = loss_fn(outputs, labels)
        accum_train_loss += loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss_data.append(float(accum_train_loss/len(train_loader.dataset)))

    with torch.no_grad():
        accum_val_loss = 0
        for imgs, labels in val_loader:
            batch_size = imgs.shape[0]
            outputs = model(imgs.view(batch_size,-1))
            loss = loss_fn(outputs, labels)
            accum_val_loss += loss
        val_loss_data.append(float(accum_val_loss/len(val_loader.dataset)))
        
    print("Epoch: %d, Train Loss: %f, Validation Loss: %f" % (epoch, accum_train_loss, accum_val_loss))

In [ ]:
plt.figure()
plt.plot(train_loss_data,label = "Train")
plt.plot(val_loss_data, label = "Validation")
plt.xlabel("Epochs")
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# check the number of parameters
numel_list = [p.numel() for p in model.parameters() if p.requires_grad == True]

sum(numel_list), numel_list

In [ ]:
# Let us try a deeper model!

model = nn.Sequential(
    nn.Linear(3072, 2),
    nn.Tanh(),
    nn.Linear(2,2),
    nn.LogSoftmax(dim=1) # LogSoftmax is more numerically stable than Softmax
)

# define the loss
loss = nn.NLLLoss()

### Optimization ###
learning_rate = 1e-2

optimizer = optim.SGD(
    model.parameters(),
    lr = learning_rate
)

loss_fn = nn.NLLLoss()

n_epochs = 200

# DataLoader class helps with constructing batches from the full Dataset class
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=100, shuffle = True)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=100, shuffle = False)

train_loss_data = []
val_loss_data = []

for epoch in range(n_epochs):
    accum_train_loss = 0
    for imgs, labels in train_loader:
        # calculate train loss
        batch_size = imgs.shape[0]
        outputs = model(imgs.view(batch_size,-1))
        loss = loss_fn(outputs, labels)
        accum_train_loss += loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss_data.append(float(accum_train_loss/len(train_loader.dataset)))

    accum_val_loss = 0
    for imgs, labels in val_loader:
        batch_size = imgs.shape[0]
        outputs = model(imgs.view(batch_size,-1))
        loss = loss_fn(outputs, labels)
        accum_val_loss += loss
    val_loss_data.append(float(accum_val_loss/len(val_loader.dataset)))
        
    print("Epoch: %d, Train Loss: %f, Validation Loss: %f" % (epoch, accum_train_loss, accum_val_loss))

In [ ]:
plt.figure()
plt.plot(train_loss_data,label = "Train")
plt.plot(val_loss_data, label = "Validation")
plt.xlabel("Epochs")
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# check the number of parameters
numel_list = [p.numel() for p in model.parameters() if p.requires_grad == True]

sum(numel_list), numel_list